<a href="https://colab.research.google.com/github/ImGabreuw/mackenzie-ai-projects/blob/master/a1_rule_based_programming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="http://meusite.mackenzie.br/rogerio/mackenzie_logo/UPM.2_horizontal_vermelho.jpg"  width=300, align="right">
<br>
<br>
<br>
<br>
<br>

# **Expert Systems, Rule‑Based Programming `PyKnown`**
---


In [12]:
#@title **Identificação do Grupo**

#@markdown Integrantes do Grupo, nome completo em orgem alfabética (*informe \<RA\>,\<nome\>*)
Aluno1 = '10418358, Bruna Aguiar Muchiuti  ' #@param {type:"string"}
Aluno2 = '10418247, Gabriel Ken Kazama Geronazzo' #@param {type:"string"}
Aluno3 = '10418013, Lucas Pires de Camargo Sarai' #@param {type:"string"}
Aluno4 = '10410798, Jessica dos Santos Santana Bispo ' #@param {type:"string"}
Aluno5 = '10409053, Otávio Augusto Freire de Andrade Bruzadin ' #@param {type:"string"}
Aluno6 = 'none' #@param {type:"string"}


# **Resumo**

Neste trabalho, foi desenvolvido um sistema especialista que sugere medicamentos com base em sintomas informados pelo usuário. O sistema utiliza regras para analisar os sintomas e indicar possíveis tratamentos.

Primeiro, os sintomas são organizados, depois o sistema identifica o tipo de problema e, por fim, sugere medicamentos adequados. Em alguns casos, pode indicar mais de uma opção, já que diferentes remédios podem servir para os mesmos sintomas.

O objetivo do trabalho é demonstrar, de forma simples, como regras podem ser usadas para tomar decisões em um sistema computacional.

# **Implementação**




In [13]:
!pip uninstall yfinance # incompatível com o pyknow

In [14]:
import sys
!{sys.executable} -m pip install git+https://github.com/buguroo/pyknow/

  Cloning https://github.com/buguroo/pyknow/ to /tmp/pip-req-build-40fs5jms
  Running command git clone --filter=blob:none --quiet https://github.com/buguroo/pyknow/ /tmp/pip-req-build-40fs5jms
  Resolved https://github.com/buguroo/pyknow/ to commit 48818336f2e9a126f1964f2d8dc22d37ff800fe8
  Preparing metadata (setup.py) ... done


In [15]:
import collections.abc
collections.Mapping = collections.abc.Mapping

from pyknow import *


# **Explicação Geral**
A implementação foi organizada em três partes principais:

A primeira parte corresponde à normalização dos sintomas. Nela, diferentes formas de descrever um mesmo problema são transformadas em fatos padronizados. Por exemplo, “febre” e “temperatura alta” passam a ser tratados como “tem febre”.

A segunda parte é composta por regras de inferência intermediária, responsáveis por identificar necessidades terapêuticas a partir da combinação dos sintomas. Assim, o sistema não recomenda medicamentos diretamente a partir dos sintomas brutos, mas passa antes por uma etapa intermediária de interpretação lógica.

A terceira parte reúne as regras de recomendação de medicamentos, que utilizam os fatos inferidos para chegar às sugestões finais.

# **Estrutura das regras**
As regras da base de conhecimento utilizam os operadores lógicos **AND, OR e NOT.**

- **OR** é usado quando diferentes sintomas representam uma mesma condição.
- **AND** é usado quando a inferência depende da presença simultânea de mais de um fato.
- **NOT** é usado quando a ausência de um fato também interfere na decisão.

Dessa forma, mostramos que o sistema inclui tanto regras simples quanto regras compostas.

In [16]:
class Medicines(KnowledgeEngine):

    @Rule(OR(Fact('febre'), Fact('temperatura alta')))
    def fever_case(self):
        self.declare(Fact('tem febre'))

    @Rule(OR(Fact('dor de cabeça'), Fact('enxaqueca')))
    def headache_case(self):
        self.declare(Fact('tem dor de cabeça'))

    @Rule(OR(Fact('dor de barriga'), Fact('dor abdominal'), Fact('colica')))
    def stomach_case(self):
        self.declare(Fact('tem dor de barriga'))

    @Rule(OR(Fact('dor no corpo'), Fact('dor muscular'), Fact('dor nas articulacoes')))
    def body_pain_case(self):
        self.declare(Fact('tem dor no corpo'))

    @Rule(OR(Fact('azia'), Fact('refluxo')))
    def acidity_case(self):
        self.declare(Fact('tem acidez'))

    @Rule(OR(Fact('diarreia'), Fact('evacuacao liquida')))
    def intestinal_case(self):
        self.declare(Fact('problema intestinal'))

    @Rule(OR(Fact('nausea'), Fact('vomito')))
    def nausea_case(self):
        self.declare(Fact('tem nausea'))

    @Rule(OR(Fact('coriza'), Fact('espirro'), Fact('nariz irritado')))
    def allergy_case(self):
        self.declare(Fact('irritacao nasal'))

    @Rule(OR(Fact('tosse'), Fact('dor de garganta')))
    def respiratory_case(self):
        self.declare(Fact('problema respiratorio'))

    @Rule(Fact('inchaco'))
    def inflammation_case(self):
        self.declare(Fact('possivel inflamacao'))

    @Rule(AND(
        Fact('tem febre'),
        OR(Fact('tem dor de cabeça'), Fact('tem dor no corpo'))
    ))
    def analgesic_antipyretic(self):
        self.declare(Fact('precisa analgesico e antitermico'))

    @Rule(AND(
        Fact('tem dor de cabeça'),
        NOT(Fact('tem febre'))
    ))
    def analgesic(self):
        self.declare(Fact('precisa analgesico'))

    @Rule(AND(
        Fact('tem dor no corpo'),
        NOT(Fact('tem febre'))
    ))
    def pain_relief(self):
        self.declare(Fact('precisa remedio para dor'))

    @Rule(AND(
        Fact('possivel inflamacao'),
        OR(Fact('tem dor no corpo'), Fact('tem dor de barriga'))
    ))
    def anti_inflammatory(self):
        self.declare(Fact('precisa antiinflamatorio'))

    @Rule(AND(
        Fact('tem acidez'),
        OR(Fact('tem dor de barriga'), Fact('queimacao no estomago'))
    ))
    def antacid(self):
        self.declare(Fact('precisa antiacido'))

    @Rule(AND(
        Fact('problema intestinal'),
        Fact('tem dor de barriga')
    ))
    def antidiarrheal(self):
        self.declare(Fact('precisa antidiarreico'))

    @Rule(AND(
        Fact('tem nausea'),
        OR(Fact('vomito'), Fact('enjoo'))
    ))
    def antiemetic(self):
        self.declare(Fact('precisa antiemetico'))

    @Rule(AND(
        Fact('irritacao nasal'),
        Fact('coceira')
    ))
    def antihistamine(self):
        self.declare(Fact('precisa antihistaminico'))

    @Rule(AND(
        Fact('problema respiratorio'),
        Fact('tosse')
    ))
    def cough_medicine(self):
        self.declare(Fact('precisa remedio para tosse'))

    @Rule(AND(
        Fact('nariz entupido'),
        OR(Fact('coriza'), Fact('pressao nos seios da face'))
    ))
    def decongestant(self):
        self.declare(Fact('precisa descongestionante'))

    @Rule(AND(
        Fact('tem febre'),
        Fact('problema respiratorio'),
        Fact('coriza')
    ))
    def cold_flu(self):
        self.declare(Fact('precisa remedio para gripe'))

    @Rule(Fact('precisa analgesico e antitermico'))
    def paracetamol(self):
        self.declare(Fact(remedio='paracetamol'))

    @Rule(AND(
        Fact('precisa analgesico'),
        Fact('tem dor de cabeça'),
        NOT(Fact('tem febre'))
    ))
    def dipyrone(self):
        self.declare(Fact(remedio='dipirona'))

    @Rule(AND(
        Fact('precisa remedio para dor'),
        Fact('possivel inflamacao')
    ))
    def ibuprofen(self):
        self.declare(Fact(remedio='ibuprofeno'))

    @Rule(AND(
        Fact('precisa antiinflamatorio'),
        Fact('dor nas articulacoes')
    ))
    def naproxen(self):
        self.declare(Fact(remedio='naproxeno'))

    @Rule(AND(
        Fact('precisa antiacido'),
        Fact('azia')
    ))
    def magnesium_hydroxide(self):
        self.declare(Fact(remedio='hidroxido de magnesio'))

    @Rule(AND(
        Fact('precisa antiacido'),
        Fact('refluxo')
    ))
    def omeprazole(self):
        self.declare(Fact(remedio='omeprazol'))

    @Rule(AND(
        Fact('precisa antidiarreico'),
        NOT(Fact('tem febre'))
    ))
    def loperamide(self):
        self.declare(Fact(remedio='loperamida'))

    @Rule(Fact('precisa antiemetico'))
    def ondansetron(self):
        self.declare(Fact(remedio='ondansetrona'))

    @Rule(Fact('precisa antihistaminico'))
    def loratadine(self):
        self.declare(Fact(remedio='loratadina'))

    @Rule(AND(
        Fact('precisa remedio para tosse'),
        Fact('tosse seca')
    ))
    def dextromethorphan(self):
        self.declare(Fact(remedio='dextrometorfano'))

    @Rule(AND(
        Fact('precisa remedio para tosse'),
        Fact('catarro')
    ))
    def expectorant(self):
        self.declare(Fact(remedio='guaifenesina'))

    @Rule(Fact('precisa descongestionante'))
    def nasal_decongestant(self):
        self.declare(Fact(remedio='descongestionante nasal'))

    @Rule(Fact('precisa remedio para gripe'))
    def flu_medicine(self):
        self.declare(Fact(remedio='antigripal'))

    @Rule(Fact(remedio=MATCH.r))
    def show_result(self, r):
        print('Remédio sugerido:', r)

    def factz(self, facts_list):
        for fact in facts_list:
            self.declare(fact)

A classe `Medicines` herda de `KnowledgeEngine`, que representa o motor de inferência do sistema especialista.

As regras iniciais tratam da padronização dos sintomas, permitindo que diferentes expressões levem à mesma representação lógica.

Em seguida, são aplicadas regras intermediárias que inferem necessidades como “precisa antiácido”, “precisa analgésico” ou “precisa antiemético”. Por fim, outras regras associam essas necessidades a medicamentos específicos.

A regra `show_result` imprime cada medicamento sugerido pelo sistema. Já o método `factz` foi criado para facilitar a inserção de vários fatos em sequência.

# **Resultados (Testes)**



### **Teste 1**
Neste teste foram inseridos os **sintomas febre, dor de cabeça e dor no corpo.** A expectativa é que o seja identificado um quadro compatível com necessidade de **analgésico e antitérmico.**

Como consequência, o sistema sugere **paracetamol e dipirona.**

In [17]:
ex1 = Medicines()
ex1.reset()
ex1.factz([
    Fact('febre'),
    Fact('dor de cabeça'),
    Fact('dor no corpo')
])
ex1.run()

Remédio sugerido: dipirona
Remédio sugerido: paracetamol


### **Teste 2**
Neste teste são inseridos os fatos **refluxo e dor abdominal.** O objetivo é verificar se o sistema consegue identificar um caso compatível com acidez gástrica e sugerir um medicamento adequado.

O fato **refluxo** é convertido em tem **acidez**, enquanto **dor abdominal** é normalizado para tem **dor de barriga**. Em seguida, a base infere a necessidade de um **antiacido**.

Como o sintoma inicial envolve refluxo, a recomendação final esperada é **omeprazol**.

In [18]:
ex2 = Medicines()
ex2.reset()
ex2.factz([
    Fact('refluxo'),
    Fact('dor abdominal')
])
ex2.run()

Remédio sugerido: omeprazol


### **Teste 3**

Neste teste são informados os sintomas **diarreia** e **dor de barriga**. O objetivo é analisar o comportamento do sistema em um **quadro intestinal simples**.

A partir desses fatos, o sistema deve inferir **problema intestinal** e tem **dor de barriga**, concluindo então que o caso precisa **antidiarreico**. Como não há febre entre os fatos iniciais, a regra da **loperamida** pode ser ativada.

In [19]:
ex3 = Medicines()
ex3.reset()
ex3.factz([
    Fact('diarreia'),
    Fact('dor de barriga')
])
ex3.run()

Remédio sugerido: loperamida


### **Teste 4**

Como teste adicional, foi considerado um quadro com **tosse** e **tosse seca**.

Primeiro, a presença de tosse leva ao fato **problema respiratorio**. Em seguida, o sistema conclui que precisa remedio para tosse. Como foi informado especificamente **tosse seca,** a recomendação esperada é **dextrometorfano**.

In [20]:
ex4 = Medicines()
ex4.reset()
ex4.factz([
    Fact('tosse'),
    Fact('tosse seca')
])
ex4.run()

Remédio sugerido: dextrometorfano


# **Referências**

Materiais de aula. Disponível em: https://github.com/Rogerio-mack/IA_2026S1. Acesso em: 18 mar. 2026.

BUGUROO. PyKnow. GitHub, [s.d.]. Disponível em: https://github.com/buguroo/pyknow/. Acesso em: 19 mar. 2026.

PYTHON SOFTWARE FOUNDATION. Python Documentation. [S. l.], [s.d.]. Disponível em: https://docs.python.org/3/. Acesso em: 19 mar. 2026.


In [23]:
#@title **Avaliação**
Resumo = 10 #@param {type:"slider", min:0, max:10, step:1}

Implementacao = 10 #@param {type:"slider", min:0, max:10, step:1}

Resultados = 10 #@param {type:"slider", min:0, max:10, step:1}

Referencias = 10 #@param {type:"slider", min:0, max:10, step:1}

Geral = 10 #@param {type:"slider", min:0, max:10, step:1}








In [24]:
#@title **Nota Final**
nota = Resumo + Implementacao + Resultados + Referencias + Geral

nota = nota / 5

print(f'Nota final do trabalho {nota :.1f}')

import numpy as np
import pandas as pd

alunos = pd.DataFrame()

lista_tia = []
lista_nome = []

for i in range(1,7):
  alumno_variable_name = "Aluno" + str(i)
  # Access the variable's value from the global scope
  alumno_value = globals()[alumno_variable_name]

  # Check if the value is not empty and contains a comma to ensure it's a valid entry
  if alumno_value and ',' in alumno_value:
    lista = alumno_value.split(',')
    # Ensure there are at least two elements after splitting
    if len(lista) >= 2:
      lista_tia.append(lista[0].strip())
      lista_nome.append(lista[1].strip().upper())

alunos['tia'] = lista_tia
alunos['nome'] = lista_nome
alunos['nota'] = np.round(nota,1)
print()
display(alunos)

Nota final do trabalho 10.0



,tia,nome,nota
0,10418358,BRUNA AGUIAR MUCHIUTI,10.0
1,10418247,GABRIEL KEN KAZAMA GERONAZZO,10.0
2,10418013,LUCAS PIRES DE CAMARGO SARAI,10.0
3,10410798,JESSICA DOS SANTOS SANTANA BISPO,10.0
4,10409053,OTÁVIO AUGUSTO FREIRE DE ANDRADE BRUZADIN,10.0
